# Day 4: R-to-Python Translation

Goal: Translate common tidyverse workflows into pandas.

Today we practice:

- `mutate()` → `.assign()` or creating new columns
- `filter()` → boolean filtering
- `arrange()` → `.sort_values()`
- `group_by()` + `summarise()` → `.groupby().agg()`
- `left_join()` → `.merge(..., how="left")`
- `pivot_longer()` → `.melt()`
- `pivot_wider()` → `.pivot()` / `.pivot_table()`

The business logic stays the same. Only the syntax changes.

In [ ]:
import pandas as pd

In [ ]:
customers = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004", "C005"],
    "customer_name": ["Anna", "Ben", "Clara", "David", "Eva"],
    "country": ["Germany", "Germany", "France", "Spain", "Germany"],
    "segment": ["Consumer", "Business", "Consumer", "Business", "Consumer"]
})

orders = pd.DataFrame({
    "order_id": ["O001", "O002", "O003", "O004", "O005", "O006"],
    "customer_id": ["C001", "C001", "C002", "C003", "C999", "C005"],
    "order_date": ["2026-01-05", "2026-01-20", "2026-02-03", "2026-02-15", "2026-03-01", "2026-03-12"],
    "sales_channel": ["Online", "Store", "Online", "Partner", "Online", "Store"],
    "order_value": [120, 80, 250, 300, 90, 150],
    "status": ["Delivered", "Returned", "Delivered", "Delivered", "Delivered", "Cancelled"]
})

monthly_sales = pd.DataFrame({
    "month": ["2026-01", "2026-02", "2026-03"],
    "Online": [120, 250, 90],
    "Store": [80, 0, 150],
    "Partner": [0, 300, 0]
})

customers

In [ ]:
orders

Translation from R to Python:

A.
mutate() → create new columns

In [ ]:
in R:

orders %>%
  mutate(
    is_completed = status == "Delivered",
    revenue = if_else(status == "Delivered", order_value, 0)
  )

In [ ]:
in pandas:

orders_with_revenue = orders.assign(
    is_completed = orders["status"] == "Delivered",
    revenue = orders["order_value"].where(orders["status"] == "Delivered", 0)
)

orders_with_revenue

# .assign() creates new columns, similar to mutate().
# the .where() part: use order_value where the order was Delivered, otherwise use 0.

B.
filter() → boolean filtering

In [ ]:
in R:

orders_with_revenue %>%
  filter(status == "Delivered", order_value >= 100)

In [ ]:
in pandas:

high_value_delivered_orders = orders_with_revenue[
    (orders_with_revenue["status"] == "Delivered") &
    (orders_with_revenue["order_value"] >= 100)
]

high_value_delivered_orders

'''
Each condition goes inside parentheses.
Use & for AND.
Use | for OR.
Use ~ for NOT.
'''

# example:

not_cancelled_orders = orders_with_revenue[
    orders_with_revenue["status"] != "Cancelled"
]

not_cancelled_orders